# MFU / HFU 的口径实验

> 对应博客篇目：《[性能剖析（00）：度量口径——MFU、HFU、Timer 与 goodput，先把尺子做准](https://lrypcy.github.io/2026/09/29/profiling-00-metrics/)》
>
> 代码逐字摘自博客仓库 `tools/profiling_bench/mfu_accounting.py` 与 `tools/megatron_bench/`（[lrypcy/ipynbs](https://github.com/lrypcy/ipynbs)）。
> 全部纯 CPU，只需要 numpy 栈里的标准库部分，秒级跑完。

## 这个 notebook 回答什么

同一行训练日志，可以合法地报成 20.2% 或 128.2% 的 MFU。差异全部来自分母：用的是稠密还是带 2:4 稀疏的规格、是 BF16 还是 FP8、是不是同一代卡。

**MFU 不是一个数，是「分子口径 + 分母口径 + 硬件」的三元组。** 报 MFU 不写分母，等于没报。

本 notebook 分六步把这件事拆开：

| 步骤 | 问题 |
|:---|:---|
| 1 | 分子固定，只换分母，MFU 差多少 |
| 2 | 分子换 recompute 策略，MFU 与 HFU 分道扬镳多少 |
| 3 | 6ND 近似随序列变长失真多少 |
| 4 | 换一代卡，机器平衡点抬高多少 |
| 5 | 一次 forward 的 FLOPs 构成里有多少不是 GEMM |
| 6 | 把官方 log 的 TFLOP/s/GPU 换回 MFU |

## 第 0 步 · 环境与共享记账

**这步在做什么**：把三个依赖摊平到 notebook 里，之后每个实验单元都是纯粹的「口径对比」。

- `configs`：模型与硬件的标称规格（Llama-3-8B / A100 / H100 / B200），每个数字都带出处。
- `flops`：FLOPs 记账，区分**逐算子精确**与**6ND 近似**两种口径。

`forward_flops_per_token()` 返回的是**单层**的量；乘上 `n_layers` 才是完整模型。lm_head 每次 forward 只算一次，所以它不乘层数——这一点在第 5 步会专门验证。

In [1]:
from dataclasses import dataclass, field
from typing import Dict, Optional
from dataclasses import dataclass
from typing import Dict

In [2]:
# ---- 摘自 tools/megatron_bench/configs.py ----
@dataclass
class ModelSpec:
    """一个 dense Transformer（GPT 系）的规格。"""

    name: str
    hidden: int          # h
    n_layers: int        # L
    n_heads: int         # attention heads
    ffn_hidden: int      # I, SwiGLU 的中间宽度（已经是乘过 ffn_mult 的值）
    vocab: int
    seq_len: int
    n_kv_heads: Optional[int] = None   # None 表示 MHA；否则是 GQA 的 kv 组数
    gated_mlp: bool = True             # True=SwiGLU(gate+up+down 三个矩阵)；False=GELU(up+down 两个)
    source: str = ""

    def __post_init__(self):
        if self.n_kv_heads is None:
            self.n_kv_heads = self.n_heads
        assert self.hidden % self.n_heads == 0, "hidden 必须能被 head 数整除"

    @property
    def head_dim(self) -> int:
        return self.hidden // self.n_heads

    @property
    def kv_dim(self) -> int:
        return self.n_kv_heads * self.head_dim

    def n_params(self, untie_embeddings: bool = False) -> int:
        """非 embedding 参数量 + embedding。用于和官方公布的参数量对账。"""
        attention = self.hidden * self.hidden          # W_q
        attention += 2 * self.hidden * self.kv_dim     # W_k, W_v
        attention += self.hidden * self.hidden         # W_o
        if self.gated_mlp:
            mlp = 2 * self.hidden * self.ffn_hidden    # gate + up
            mlp += self.ffn_hidden * self.hidden       # down
        else:
            mlp = 2 * self.hidden * self.ffn_hidden    # up + down
        per_layer = attention + mlp
        embed = self.vocab * self.hidden
        total = self.n_layers * per_layer + embed
        if not untie_embeddings:
            total += embed                             # output layer 独立
        return total

    def n_params_no_embedding(self) -> int:
        attention = 2 * self.hidden * self.hidden + 2 * self.hidden * self.kv_dim
        mlp = self.hidden * self.ffn_hidden * (3 if self.gated_mlp else 2)
        return self.n_layers * (attention + mlp)



MODELS: Dict[str, ModelSpec] = {
    # Llama-2 7B 结构：hidden 4096 / 32 层 / MHA / SwiGLU 中间 11008
    "llama-7b": ModelSpec(
        name="llama-7b", hidden=4096, n_layers=32, n_heads=32,
        ffn_hidden=11008, vocab=32000, seq_len=4096,
        source="Llama-2-7B config (HuggingFace meta-llama/Llama-2-7b-hf)",
    ),
    # Llama-3 8B 系：改了 GQA（8 个 kv head）+ 更大词表
    "llama3-8b": ModelSpec(
        name="llama3-8b", hidden=4096, n_layers=32, n_heads=32, n_kv_heads=8,
        ffn_hidden=14336, vocab=128256, seq_len=8192,
        source="Llama-3-8B config (meta-llama/Meta-Llama-3-8B)",
    ),
    # Qwen2.5-14B 系 calcifications（含 GQA）
    "qwen-14b": ModelSpec(
        name="qwen-14b", hidden=5120, n_layers=48, n_heads=40, n_kv_heads=8,
        ffn_hidden=13824, vocab=152064, seq_len=8192,
        source="Qwen2.5-14B config (Qwen/Qwen2.5-14B)",
    ),
    # GPT-3 175B：Megatron-LM examples/pretrain_gpt3_175B.sh
    "gpt3-175b": ModelSpec(
        name="gpt3-175b", hidden=12288, n_layers=96, n_heads=96,
        ffn_hidden=49152, vocab=50257, seq_len=2048, gated_mlp=False,
        source="Megatron-LM examples/gpt3/train_gpt3_175b_distributed.sh",
    ),
}



@dataclass
class HardwareSpec:
    """一张卡的标称 specs。带宽统一用单向 GB/s，避免 bidir 口径混淆。"""

    name: str
    bf16_tflops: float      # dense BF16 算力峰值 (TFLOP/s)
    hbm_gbs: float          # HBM 带宽 (GB/s)
    mem_gb: float           # 单卡显存 (GiB)
    nvlink_gbs: float       # NVLink 单向聚合带宽 (GB/s)
    ib_gbs: float           # 节点间网络单向带宽 (GB/s)
    gpus_per_node: int
    source: str = ""



HARDWARE: Dict[str, HardwareSpec] = {
    "a100-80g": HardwareSpec(
        name="a100-80g", bf16_tflops=312.0, hbm_gbs=2039.0, mem_gb=80.0,
        nvlink_gbs=300.0, ib_gbs=25.0, gpus_per_node=8,
        source="NVIDIA A100 datasheet (BF16 dense 312 TFLOP/s, HBM2e 2.0 TB/s)",
    ),
    "h100-80g": HardwareSpec(
        name="h100-80g", bf16_tflops=989.0, hbm_gbs=3350.0, mem_gb=80.0,
        nvlink_gbs=450.0, ib_gbs=50.0, gpus_per_node=8,
        source="NVIDIA H100 SXM datasheet (BF16 dense 989 TFLOP/s, HBM3 3.35 TB/s)",
    ),
    "b200-180g": HardwareSpec(
        name="b200-180g", bf16_tflops=2250.0, hbm_gbs=8000.0, mem_gb=180.0,
        nvlink_gbs=900.0, ib_gbs=100.0, gpus_per_node=8,
        source="NVIDIA B200 datasheet (BF16 dense 2.25 PFLOP/s, HBM3e 8 TB/s)",
    ),
}



@dataclass
class BenchConfig:
    """一次求解的完整输入。"""

    model: str = "llama-7b"
    hardware: str = "h100-80g"
    tp: int = 1
    cp: int = 1
    pp: int = 1
    ep: int = 1
    dp: int = 1
    num_nodes: int = 1
    micro_batch: int = 4
    grad_accum: int = 8
    seq_len: Optional[int] = None
    recompute: str = "none"          # none | full | selective
    use_distributed_optimizer: bool = True
    bytes_per_param_elem: int = 2    # bf16 = 2, fp32 = 4
    extra_notes: list = field(default_factory=list)

    def resolved_seq_len(self) -> int:
        return self.seq_len or MODELS[self.model].seq_len

    @property
    def world_size(self) -> int:
        """总卡数 = 各并行维乘积。dp 一般是被最后算出来的那一维。"""
        return self.tp * self.cp * self.pp * self.ep * self.dp

In [3]:
# ---- 摘自 tools/megatron_bench/flops.py ----
@dataclass
class FlopsBreakdown:
    """每 token 的 forward FLOPs 明细。单位都是 FLOP/token。"""

    qkv_proj: float
    attn_core: float
    attn_out: float
    mlp: float
    lm_head: float

    def forward_total(self, include_lm_head: bool = True) -> float:
        t = self.qkv_proj + self.attn_core + self.attn_out + self.mlp
        if include_lm_head:
            t += self.lm_head
        return t

    def as_rows(self) -> list:
        return [
            ("QKV 投影", self.qkv_proj),
            ("attention core (QK^T + AV)", self.attn_core),
            ("输出投影 W_o", self.attn_out),
            ("MLP (SwiGLU 三个投影)", self.mlp),
            ("lm_head (vocab 投影)", self.lm_head),
        ]



def forward_flops_per_token(cfg: BenchConfig, causal: bool = False) -> FlopsBreakdown:
    """每 token 的 forward FLOPs。不计 elementwise（LN / RoPE / dropout / residual）。

    GEMM 类算子的 FLOPs = 2 * tokens * in * out。
    """
    m = MODELS[cfg.model]
    h, I = m.hidden, m.ffn_hidden
    kv_dim = m.kv_dim
    s = cfg.resolved_seq_len()

    qkv = 2 * h * h + 2 * 2 * h * kv_dim          # W_q: h->h, W_k/W_v: h->kv_dim
    attn_core = 4 * s * h                          # QK^T + AV，每个 token 要打 s 个位置
    if causal:
        attn_core /= 2                             # 下三角，实际乘一半
    attn_out = 2 * h * h
    # SwiGLU 有 gate/up/down 三个投影 -> 6hI；GELU 只有 up/down 两个 -> 4hI
    mlp = h * I * (6 if m.gated_mlp else 4)
    lm_head = 2 * h * m.vocab

    return FlopsBreakdown(qkv, attn_core, attn_out, mlp, lm_head)



def training_flops_per_token(cfg: BenchConfig, causal: bool = False) -> Dict[str, float]:
    """每 token 的训练总 FLOPs，按 recompute 策略分别给出。

    注意 forward_flops_per_token 返回的是**单层**的量，这里才乘上层数。
    backward = 2 x forward（权梯度 + 输入梯度两个 GEMM）。
    """
    m = MODELS[cfg.model]
    L = m.n_layers
    fwd = forward_flops_per_token(cfg, causal)

    hidden_forward = fwd.forward_total(include_lm_head=False) * L   # 所有 transformer 层
    full_forward = hidden_forward + fwd.lm_head

    out = {}
    out["forward"] = full_forward
    out["backward"] = 2 * full_forward
    out["total_none"] = 3 * full_forward
    # full recompute：反向阶段把所有 transformer 层再算一遍 forward
    out["total_full_recompute"] = 3 * full_forward + hidden_forward
    # selective(core_attn)：只重算 attention core
    out["total_selective_core_attn"] = 3 * full_forward + fwd.attn_core * L
    out["_hidden_forward_per_layer"] = fwd.forward_total(include_lm_head=False)
    out["_attn_core"] = fwd.attn_core
    return out



def approx_6nd(cfg: BenchConfig) -> float:
    """6ND 近似：N 取参与 GEMM 的参数量（不含 embedding 与 lm_head）。"""
    m = MODELS[cfg.model]
    return 6 * m.n_params_no_embedding()

In [4]:
# ---- 模块名垫片：让原脚本里 `configs.X` / `flops.X` 的写法原样可用 ----
import types

configs = types.SimpleNamespace(BenchConfig=BenchConfig, MODELS=MODELS,
                                HARDWARE=HARDWARE)
flops = types.SimpleNamespace(forward_flops_per_token=forward_flops_per_token,
                              training_flops_per_token=training_flops_per_token,
                              approx_6nd=approx_6nd)

## 第 1 步 · 分母口径：同一行 log，换个分母就换一个 MFU

**这步在做什么**：分子完全固定（同一份每 token FLOPs），只把分母在六种规格之间换来换去。

入口取的是「官方 log 那一行」的 TFLOP/s/GPU，而不是自己编吞吐——因为这正是读者会遇到的那个数。400 TFLOP/s/GPU 对 8B 模型在 H100 上是常见量级（≈40% MFU）。

峰值来源均为官方产品页（核验日期 2026-09-29）：

- [A100 产品页](https://www.nvidia.com/en-us/data-center/a100/)：BF16 312 TFLOPS（稠密）| 624 TFLOPS\*（稀疏），HBM 2039 GB/s
- [H100 产品页](https://www.nvidia.com/en-us/data-center/h100/)：BF16 1979 TFLOPS\*（= 稠密 989.5），FP8 3958\*（= 稠密 1979），HBM 3.35 TB/s

注意两个页面的排版口径并不一致：A100 把稠密值放前面，H100 只给带 \* 的稀疏值。**照抄首页数字当分母，MFU 直接腰斩。**

In [5]:
# ---- 摘自 tools/profiling_bench/mfu_accounting.py ----
PEAKS = {
    "A100 80GB SXM · BF16 稠密": 312.0,
    "A100 80GB SXM · BF16 稀疏 2:4": 624.0,
    "H100 SXM · BF16 稠密": 989.5,
    "H100 SXM · BF16 稀疏 2:4": 1979.0,
    "H100 SXM · FP8 稠密": 1979.0,
    "H100 SXM · TF32 稠密": 494.5,
}



GEN = {
    "A100 80GB SXM": {"bf16_dense": 312.0, "hbm_gbs": 2039.0},
    "H100 SXM": {"bf16_dense": 989.5, "hbm_gbs": 3350.0},
}



def _cfg(model="llama3-8b", **kw):
    base = dict(model=model, hardware="h100-80g", tp=1, cp=1, pp=1, ep=1, dp=8,
                num_nodes=1, micro_batch=4, grad_accum=8, seq_len=None,
                recompute="none", use_distributed_optimizer=True)
    base.update(kw)
    return configs.BenchConfig(**base)



def exp_denominator(model="llama3-8b", seq_len=8192, tf_per_gpu=400.0):
    """实验一：分子固定，只换分母 -> MFU 差多少。

    入口取「官方 log 那一行的 TFLOP/s/GPU」，避免自己编吞吐。
    400 TFLOP/s/GPU 对 8B 模型在 H100 上是常见量级（≈40% MFU）。
    """
    cfg = _cfg(model=model, seq_len=seq_len)
    per_token = flops.training_flops_per_token(cfg)["total_none"]
    tps = tf_per_gpu * 1e12 / per_token

    print("=" * 78)
    print("实验一  分母口径：同一行 log，换个分母就换一个 MFU")
    print("=" * 78)
    print("模型 %s，序列 %d，每 token %.2f GFLOP（不含重计算）" % (model, seq_len, per_token / 1e9))
    print("log 读数 = %.0f TFLOP/s/GPU  ->  折合 %.0f tokens/s/GPU"
          % (tf_per_gpu, tps))
    print("分子固定不变，只换分母：")
    print()
    print("  分母（TFLOPS）                         MFU     相对 H100 稠密 BF16")
    ref = None
    for name, peak in PEAKS.items():
        m = tf_per_gpu / peak
        if "H100 SXM · BF16 稠密" in name:
            ref = m
        print("  %-36s %7.2f%%" % (name, 100 * m))
    print()
    lo, hi = min(PEAKS.values()), max(PEAKS.values())
    print("  最大 / 最小分母之比 = %.2fx -> 同一行 log 可以报成 %.1f%% 或 %.1f%%"
          % (hi / lo, 100 * tf_per_gpu / hi, 100 * tf_per_gpu / lo))
    print()
    print("  注意 A100 那两行给出了 >100%% 的「MFU」。这个数本身就是一个诊断：")
    print("  400 TFLOP/s/GPU 超过了 A100 稠密 BF16 的峰值 312，所以这行日志不可能来自 A100。")
    print("  跨硬件套错分母，得到的是一个物理上不存在的利用率。")
    print()
    print("=> MFU 没有绝对值，只有「分子口径 + 分母口径 + 硬件」三元组。")
    print("   报 MFU 不写分母用的是稠密还是稀疏、哪种精度，等于没报。")
    print()



exp_denominator()

实验一  分母口径：同一行 log，换个分母就换一个 MFU
模型 llama3-8b，序列 8192，每 token 57.91 GFLOP（不含重计算）
log 读数 = 400 TFLOP/s/GPU  ->  折合 6907 tokens/s/GPU
分子固定不变，只换分母：

  分母（TFLOPS）                         MFU     相对 H100 稠密 BF16
  A100 80GB SXM · BF16 稠密               128.21%
  A100 80GB SXM · BF16 稀疏 2:4            64.10%
  H100 SXM · BF16 稠密                     40.42%
  H100 SXM · BF16 稀疏 2:4                 20.21%
  H100 SXM · FP8 稠密                      20.21%
  H100 SXM · TF32 稠密                     80.89%

  最大 / 最小分母之比 = 6.34x -> 同一行 log 可以报成 20.2% 或 128.2%

  注意 A100 那两行给出了 >100%% 的「MFU」。这个数本身就是一个诊断：
  400 TFLOP/s/GPU 超过了 A100 稠密 BF16 的峰值 312，所以这行日志不可能来自 A100。
  跨硬件套错分母，得到的是一个物理上不存在的利用率。

=> MFU 没有绝对值，只有「分子口径 + 分母口径 + 硬件」三元组。
   报 MFU 不写分母用的是稠密还是稀疏、哪种精度，等于没报。



> **这一步的结论**：A100 那两行给出了 >100% 的「MFU」。这个数本身就是一个诊断——400 TFLOP/s/GPU 超过了 A100 稠密 BF16 的峰值 312，所以这行日志不可能来自 A100。**跨硬件套错分母，得到的是一个物理上不存在的利用率。**

## 第 2 步 · 分子口径：MFU 不变而 HFU 变，差额就是重计算的代价

**这步在做什么**：分母固定成 H100 稠密 BF16，只改 recompute 策略。

关键在于两个指标的分子定义不同：

- **MFU** 的分子收尾在 Megatron `training.py:1007` 的 `return flops_fwd * 3`，**不含重计算系数**——所以开不开 recompute 这个数都不变。
- **HFU** 按实际发射量算，重算一遍就多一遍。

开满 full recompute 时每 token 从 $6P$ 变成 $8P$，比值恒为 $4/3$。**想评价「重计算划不划算」得自己换算成 HFU，拿 MFU 去问是问错了指标。**

In [6]:
# ---- 摘自 tools/profiling_bench/mfu_accounting.py ----
def exp_numerator(model="llama3-8b", seq_len=8192, tf_per_gpu=400.0):
    """实验二：分子换 recompute 策略 -> MFU 与 HFU 分道扬镳。"""
    cfg = _cfg(model=model, seq_len=seq_len)
    t = flops.training_flops_per_token(cfg)
    peak = PEAKS["H100 SXM · BF16 稠密"]

    print("=" * 78)
    print("实验二  分子口径：MFU 不变而 HFU 变，差额就是重计算的代价")
    print("=" * 78)
    print("分母统一用 H100 稠密 BF16 = %.1f TFLOPS；log 读数 %.0f TFLOP/s/GPU"
          % (peak, tf_per_gpu))
    print()
    rows = [("不重计算", t["total_none"]),
            ("full recompute", t["total_full_recompute"]),
            ("selective (core attn)", t["total_selective_core_attn"])]
    base = t["total_none"]
    print("  策略                 每 token FLOPs     MFU      HFU     HFU/MFU")
    for name, fl in rows:
        mfu = tf_per_gpu / peak                              # MFU 不随 recompute 变
        hfu = tf_per_gpu * (fl / base) / peak                # HFU 按实际发射量
        print("  %-20s %8.2f GFLOP  %6.2f%%  %6.2f%%    %.3f"
              % (name, fl / 1e9, 100 * mfu, 100 * hfu, fl / base))
    print()
    print("  full recompute 时 解析比值 = %.4f（= 4/3，每 token 从 6P 变 8P）"
          % (t["total_full_recompute"] / t["total_none"]))
    print("  selective 时     解析比值 = %.4f"
          % (t["total_selective_core_attn"] / t["total_none"]))
    print()
    print("=> Megatron 官方 log 的 `throughput per GPU (TFLOP/s/GPU)` 来自")
    print("   `num_floating_point_operations()`，收尾在 L1007 的 `return flops_fwd * 3`，")
    print("   **不含重计算系数** —— 所以它报的是 MFU 口径，开不开 recompute 这个数都不变。")
    print("   想拿 HFU 得自己乘上去；拿 MFU 去评价「重计算划不划算」是问错了指标。")
    print()



exp_numerator()

实验二  分子口径：MFU 不变而 HFU 变，差额就是重计算的代价
分母统一用 H100 稠密 BF16 = 989.5 TFLOPS；log 读数 400 TFLOP/s/GPU

  策略                 每 token FLOPs     MFU      HFU     HFU/MFU
  不重计算                    57.91 GFLOP   40.42%   40.42%    1.000
  full recompute          76.17 GFLOP   40.42%   53.17%    1.315
  selective (core attn)    62.21 GFLOP   40.42%   43.42%    1.074

  full recompute 时 解析比值 = 1.3152（= 4/3，每 token 从 6P 变 8P）
  selective 时     解析比值 = 1.0742

=> Megatron 官方 log 的 `throughput per GPU (TFLOP/s/GPU)` 来自
   `num_floating_point_operations()`，收尾在 L1007 的 `return flops_fwd * 3`，
   **不含重计算系数** —— 所以它报的是 MFU 口径，开不开 recompute 这个数都不变。
   想拿 HFU 得自己乘上去；拿 MFU 去评价「重计算划不划算」是问错了指标。



## 第 3 步 · 分子口径的来源：6ND 近似随序列变长而失真

**这步在做什么**：业界常用的 $6ND$ 近似（$N$ 取参与矩阵乘的参数量）在短序列上误差可忽略，长序列上就不行了。

原因是 $6ND$ 把 attention 的 $L^2$ 项当常数忽略了。序列越长，这一项越不能忽略，于是「同一个模型」在长序列下的 6ND-MFU 与精确-MFU 会系统性背离。

对比的两个口径都含 causal 折半，所以偏差纯粹来自 $L^2$ 项的处理方式。

In [7]:
# ---- 摘自 tools/profiling_bench/mfu_accounting.py ----
def exp_seqlen_sweep(model="llama3-8b", tok_per_s=14000.0):
    """实验三：6ND 近似随序列变长失真。"""
    cfg0 = _cfg(model=model)
    print("=" * 78)
    print("实验三  分子口径的来源：6ND 近似随序列变长而失真")
    print("=" * 78)
    print("  序列长度   精确(含 lm_head)      6ND        偏差")
    for s in (1024, 2048, 4096, 8192, 32768, 131072):
        cfg = _cfg(model=model, seq_len=s)
        exact = flops.training_flops_per_token(cfg, causal=True)["total_none"]
        approx = flops.approx_6nd(cfg)
        print("  %8d   %8.2f GFLOP   %8.2f GFLOP   %+7.2f%%"
              % (s, exact / 1e9, approx / 1e9, 100 * (exact - approx) / approx))
    print()
    print("=> 6ND 把 attention 的 L^2 项当常数忽略。序列越长，这一项越不能忽略，")
    print("   于是「同一个模型」在长序列下的 6ND-MFU 与精确-MFU 会系统性背离。")
    print()



exp_seqlen_sweep()

实验三  分子口径的来源：6ND 近似随序列变长而失真
  序列长度   精确(含 lm_head)      6ND        偏差
      1024      45.83 GFLOP      41.88 GFLOP     +9.45%
      2048      46.64 GFLOP      41.88 GFLOP    +11.37%
      4096      48.25 GFLOP      41.88 GFLOP    +15.22%
      8192      51.47 GFLOP      41.88 GFLOP    +22.91%
     32768      70.80 GFLOP      41.88 GFLOP    +69.07%
    131072     148.11 GFLOP      41.88 GFLOP   +253.68%

=> 6ND 把 attention 的 L^2 项当常数忽略。序列越长，这一项越不能忽略，
   于是「同一个模型」在长序列下的 6ND-MFU 与精确-MFU 会系统性背离。



## 第 4 步 · 代际算术强度门槛：算力涨 3.2×，带宽只涨 1.6×

**这步在做什么**：换一代卡不只是分母变大，**分子能用到的那部分也不同比例地涨**。

机器平衡点 = 峰值算力 / 峰值带宽，单位 FLOP/Byte。一段算术强度固定为 $X$ 的子层，在 A100 上要 $X$ 大于平衡点才算计算受限，换到 H100 这个门槛更高。

门槛上升的直接后果：**「换新卡后吞吐上去了、但 log 里的 TFLOP/s/GPU 没同比例涨」是正常的**，不是日志坏了。

In [8]:
# ---- 摘自 tools/profiling_bench/mfu_accounting.py ----
def exp_generation():
    """实验四：代际对照 —— 算力涨得比带宽快，MFU 必然下降。"""
    print("=" * 78)
    print("实验四  代际算术强度门槛：算力涨 3.2x，带宽只涨 1.6x")
    print("=" * 78)
    a, h = GEN["A100 80GB SXM"], GEN["H100 SXM"]
    r_flops = h["bf16_dense"] / a["bf16_dense"]
    r_bw = h["hbm_gbs"] / a["hbm_gbs"]
    print("  稠密 BF16 峰值:  %.1f -> %.1f TFLOPS   = %.2fx"
          % (a["bf16_dense"], h["bf16_dense"], r_flops))
    print("  HBM 带宽:        %.0f -> %.0f GB/s     = %.2fx"
          % (a["hbm_gbs"], h["hbm_gbs"], r_bw))
    print()
    bal_a = a["bf16_dense"] * 1e12 / (a["hbm_gbs"] * 1e9)
    bal_h = h["bf16_dense"] * 1e12 / (h["hbm_gbs"] * 1e9)
    print("  机器平衡点（峰值算力 / 峰值带宽，单位 FLOP/Byte）:")
    print("    A100 80GB SXM: %.1f FLOP/Byte" % bal_a)
    print("    H100 SXM     : %.1f FLOP/Byte" % bal_h)
    print("    门槛上升     : %.2fx" % (bal_h / bal_a))
    print()
    print("=> 门槛上升意味着：一段算术强度固定为 X 的子层，在 A100 上要 X > %.1f 才是计算受限，"
          % bal_a)
    print("   在 H100 上要 X > %.1f。原本在 A100 上勉强算「计算受限」的层，" % bal_h)
    print("   换到 H100 就滑回了访存受限 —— 峰值算力上的分子涨了，能用到的那部分没涨那么多。")
    print("   所以「换新卡后吞吐上去了、但 log 里的 TFLOP/s/GPU 没同比例涨」是正常的。")
    print()



exp_generation()

实验四  代际算术强度门槛：算力涨 3.2x，带宽只涨 1.6x
  稠密 BF16 峰值:  312.0 -> 989.5 TFLOPS   = 3.17x
  HBM 带宽:        2039 -> 3350 GB/s     = 1.64x

  机器平衡点（峰值算力 / 峰值带宽，单位 FLOP/Byte）:
    A100 80GB SXM: 153.0 FLOP/Byte
    H100 SXM     : 295.4 FLOP/Byte
    门槛上升     : 1.93x

=> 门槛上升意味着：一段算术强度固定为 X 的子层，在 A100 上要 X > 153.0 才是计算受限，
   在 H100 上要 X > 295.4。原本在 A100 上勉强算「计算受限」的层，
   换到 H100 就滑回了访存受限 —— 峰值算力上的分子涨了，能用到的那部分没涨那么多。
   所以「换新卡后吞吐上去了、但 log 里的 TFLOP/s/GPU 没同比例涨」是正常的。



## 第 5 步 · 一次完整 forward 的 FLOPs 构成：不是所有 FLOPs 都长成 GEMM 的样子

**这步在做什么**：把 Llama-3-8B 一次 forward 的每 token FLOPs 拆成五项，看比例。

这里有个容易写错的地方：`forward_flops_per_token()` 返回的是**单层**的量，但 lm_head 是每次 forward 只算一次。直接拿单层的项和 lm_head 相加会得到一个没有物理含义的比例——必须按 $L$ 折算。

值得注意的是 attention core 占的比例：它里面有一半是 softmax（elementwise）而不是 matmul，而 MFU 的分子只算 matmul。**这部分 FLOPs 一分都不计入分子，但真机上要花时间。** 序列越长，这个「计入分子但不完整」的区域越大。

In [9]:
# ---- 摘自 tools/profiling_bench/mfu_accounting.py ----
def exp_layer_mix(model="llama3-8b", seq_len=8192):
    """实验五：一次完整 forward 的 FLOPs 构成。

    注意 `forward_flops_per_token()` 返回的是**单层**的量，但 lm_head 是每次 forward
    只算一次。直接拿它和单层的项相加会得到一个没有物理含义的比例，这里按 L 折算。
    """
    cfg = _cfg(model=model, seq_len=seq_len)
    d = flops.forward_flops_per_token(cfg)
    m = configs.MODELS[model]
    L = m.n_layers

    per_layer = d.qkv_proj + d.attn_core + d.attn_out + d.mlp
    hidden_all = per_layer * L
    total = hidden_all + d.lm_head

    print("=" * 78)
    print("实验五  一次完整 forward 的 FLOPs 构成：不是所有 FLOPs 都长成 GEMM 的样子")
    print("=" * 78)
    print("模型 %s（%d 层），序列 %d，每 token 口径" % (model, L, seq_len))
    print()
    print("  组成                                    每 token FLOPs   占 forward")
    rows = [("QKV 投影 × %d 层" % L, d.qkv_proj * L),
            ("attention core (QK^T + AV) × %d 层" % L, d.attn_core * L),
            ("输出投影 W_o × %d 层" % L, d.attn_out * L),
            ("MLP (SwiGLU 三个投影) × %d 层" % L, d.mlp * L),
            ("lm_head (vocab 投影，只算一次)", d.lm_head)]
    for name, v in rows:
        print("  %-40s %10.2f MFLOP     %5.1f%%" % (name, v / 1e6, 100 * v / total))
    print("  %-40s %10.2f MFLOP     %5.1f%%" % ("合计", total / 1e6, 100.0))
    print()
    print("  attention core 占 %.1f%%，其余是各层投影；lm_head 单独占 %.1f%%。"
          % (100 * d.attn_core * L / total, 100 * d.lm_head / total))
    print("  attention core 里有一半是 softmax（elementwise）而不是 matmul —— MFU 的分子")
    print("  只算 matmul，这部分 FLOPs 一分都不计入，但真机上要花时间。")
    print("  序列越长，这个「计入分子但不完整」的区域越大。")
    print()



exp_layer_mix()

实验五  一次完整 forward 的 FLOPs 构成：不是所有 FLOPs 都长成 GEMM 的样子
模型 llama3-8b（32 层），序列 8192，每 token 口径

  组成                                    每 token FLOPs   占 forward
  QKV 投影 × 32 层                               1610.61 MFLOP       8.3%
  attention core (QK^T + AV) × 32 层           4294.97 MFLOP      22.2%
  输出投影 W_o × 32 层                             1073.74 MFLOP       5.6%
  MLP (SwiGLU 三个投影) × 32 层                   11274.29 MFLOP      58.4%
  lm_head (vocab 投影，只算一次)                     1050.67 MFLOP       5.4%
  合计                                         19304.28 MFLOP     100.0%

  attention core 占 22.2%，其余是各层投影；lm_head 单独占 5.4%。
  attention core 里有一半是 softmax（elementwise）而不是 matmul —— MFU 的分子
  只算 matmul，这部分 FLOPs 一分都不计入，但真机上要花时间。
  序列越长，这个「计入分子但不完整」的区域越大。



## 第 6 步 · 把 log 里的 TFLOP/s/GPU 换回 MFU

**这步在做什么**：把前面所有口径收成一个实用工具——给定一行 log 的读数，分别按稠密分母与稀疏分母换算。

结论很直白：同一行 log，稠密分母下 40%、稀疏分母下 20%。**报数的人如果不写分母口径，你无法判断他说的是哪个。**

In [10]:
# ---- 摘自 tools/profiling_bench/mfu_accounting.py ----
def exp_official_log(model="llama3-8b", seq_len=8192):
    """实验五：把官方 log 的数换回 MFU。"""
    cfg = _cfg(model=model, seq_len=seq_len)
    per_token = flops.training_flops_per_token(cfg)["total_none"]
    print("=" * 78)
    print("实验六  把 log 里的 TFLOP/s/GPU 换回 MFU")
    print("=" * 78)
    print("模型 %s / 序列 %d / 每 token %.2f GFLOP（不含重计算）"
          % (model, seq_len, per_token / 1e9))
    print()
    print("  log 里的数    tokens/s/GPU   MFU(稠密 BF16)   MFU(稀疏 BF16)")
    for tf in (100.0, 200.0, 300.0, 400.0):
        tps = tf * 1e12 / per_token
        print("  %6.0f TFLOP/s   %10.0f     %10.2f%%       %10.2f%%"
              % (tf, tps,
                 100 * tf / PEAKS["H100 SXM · BF16 稠密"],
                 100 * tf / PEAKS["H100 SXM · BF16 稀疏 2:4"]))
    print()
    print("=> 同一行 log，稠密分母下 40%、稀疏分母下 20%。")
    print("   报数的人如果不写分母口径，你无法判断他说的是哪个。")
    print()



exp_official_log()

实验六  把 log 里的 TFLOP/s/GPU 换回 MFU
模型 llama3-8b / 序列 8192 / 每 token 57.91 GFLOP（不含重计算）

  log 里的数    tokens/s/GPU   MFU(稠密 BF16)   MFU(稀疏 BF16)
     100 TFLOP/s         1727          10.11%             5.05%
     200 TFLOP/s         3453          20.21%            10.11%
     300 TFLOP/s         5180          30.32%            15.16%
     400 TFLOP/s         6907          40.42%            20.21%

=> 同一行 log，稠密分母下 40%、稀疏分母下 20%。
   报数的人如果不写分母口径，你无法判断他说的是哪个。



## 小结

MFU/HFU 之争不是定义之争，是口径之争。写清楚「分子含不含重计算、分母用哪种精度哪个稀疏假设」才算报了个数。

**报 MFU 不写分母等于没报。** 三个元组缺一个，这个数就没法与别人的数比较。

## 参考文献

- [NVIDIA A100 产品页](https://www.nvidia.com/en-us/data-center/a100/)
- [NVIDIA H100 产品页](https://www.nvidia.com/en-us/data-center/h100/)
- Chowdhery et al., *PaLM: Scaling Language Modeling with Pathways*, [arXiv:2204.02311](https://arxiv.org/abs/2204.02311) —— MFU 指标的出处
- Megatron-LM `training.py:802` / `:1007`（基准 commit `60e039626`，v0.20.0）